# Zain Jordan Customer 360 AI Workshop
## Class 6: MCP for Enterprise AI Tool Integration

### Class Goal
In the previous classes, we built database access, LangChain tools, a SQL agent, a RAG assistant, and a multi-agent customer-care copilot.

In this class, we introduce **MCP — Model Context Protocol**.

MCP gives us a standard way to expose tools and data to AI applications.

### Main Use Case
We will expose selected Zain Jordan database functions through a local MCP server. Then a LangChain agent will connect to that MCP server and use the tools.

### Final Demo
> Use MCP tools to analyze customer 42. Check profile, plan, churn risk, billing, complaints, usage, and value segment. Recommend the next best action.

### Boundary
This class focuses on local MCP tools with stdio transport. We are not doing advanced remote deployment, authentication, or production cloud MCP servers yet.


# 1. Learning Outcomes

By the end of this notebook, participants should be able to:

1. Explain MCP in simple business and technical language.
2. Understand LangChain tools vs MCP tools.
3. Create a local MCP server using Python.
4. Expose selected Zain Jordan database functions as MCP tools.
5. Connect LangChain to the MCP server using `MultiServerMCPClient`.
6. Load MCP tools into a LangChain agent.
7. Run an MCP-powered customer-care agent.
8. Map MCP design to capstone projects.


# 2. Concept: Python Function → LangChain Tool → MCP Tool

```text
Python Function
   ↓
LangChain Tool
   ↓
Agent Uses Tool
   ↓
MCP Server Exposes Tool
   ↓
MCP Client Loads Tool
   ↓
LangChain Agent Uses MCP Tool
```

A LangChain tool is usually local to your app or notebook. An MCP tool is exposed by an MCP server, so different AI clients can connect to it through a standard protocol.


# 3. Architecture for This Class

```text
Zain Jordan SQLite Database
        ↓
Python Database Functions
        ↓
MCP Server
        ↓
LangChain MultiServerMCPClient
        ↓
LangChain Agent
        ↓
Customer-Care Recommendation
```

We use **stdio transport**, where LangChain starts the MCP server as a local subprocess and communicates through standard input/output.


In [ ]:
%pip install -q -U "mcp[cli]" fastmcp langchain langchain-openai langchain-mcp-adapters langchain-community pandas sqlalchemy


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


# 4. Set OpenAI API Key

In Google Colab, add a secret named `OPENAI_API_KEY` and enable notebook access.


In [ ]:
try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


# 5. Upload or Locate the Zain Jordan Database

Upload the same database file used in previous classes:

`zain_customer_360_ai_demo.db`


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


In [ ]:
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

db_path_obj = Path(DB_PATH).resolve()

print("Database path:", db_path_obj)
print("File exists:", db_path_obj.exists())

conn = sqlite3.connect(str(db_path_obj), check_same_thread=False)

tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


In [ ]:
pd.read_sql_query("""
SELECT 
    customer_id,
    full_name,
    city,
    customer_segment,
    preferred_language,
    status
FROM customers
LIMIT 5;
""", conn)


# 6. Create the Zain Jordan MCP Server File

We create a separate Python file:

`zain_customer_mcp_server.py`

This file exposes selected read-only customer database functions as MCP tools.


In [ ]:
SERVER_FILE = Path("zain_customer_mcp_server.py").resolve()

server_code = "\nimport sqlite3\nimport pandas as pd\nfrom pathlib import Path\n\ntry:\n    from fastmcp import FastMCP\nexcept ImportError:\n    from mcp.server.fastmcp import FastMCP\n\nDB_PATH = Path(__DB_PATH_LITERAL__)\n\nmcp = FastMCP(\"Zain Jordan Customer 360 MCP Server\")\n\ndef get_connection():\n    return sqlite3.connect(str(DB_PATH), check_same_thread=False)\n\ndef df_to_text(df, max_rows=10):\n    if df is None or df.empty:\n        return \"No records found.\"\n    return df.head(max_rows).to_string(index=False)\n\n@mcp.tool()\ndef get_customer_profile(customer_id: int) -> str:\n    \"\"\"Get customer profile, city, segment, language, account type, and account status for a Zain Jordan customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        c.gender,\n        c.age_group,\n        c.city,\n        c.governorate,\n        c.customer_type,\n        c.customer_segment,\n        c.preferred_language,\n        c.status AS customer_status,\n        a.account_type,\n        a.account_status,\n        a.credit_limit_jod\n    FROM customers c\n    LEFT JOIN accounts a ON c.customer_id = a.customer_id\n    WHERE c.customer_id = ?;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id,))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_plan(customer_id: int) -> str:\n    \"\"\"Get current subscriptions, mobile numbers, plan name, monthly fee, data allowance, minutes, and contract details for a customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        s.subscription_id,\n        s.msisdn,\n        s.service_type,\n        s.status AS subscription_status,\n        p.plan_name,\n        p.plan_category,\n        p.monthly_fee_jod,\n        p.data_allowance_gb,\n        p.local_minutes,\n        p.international_minutes,\n        p.roaming_minutes,\n        p.sms_allowance,\n        p.technology,\n        p.contract_months\n    FROM customers c\n    JOIN subscriptions s ON c.customer_id = s.customer_id\n    JOIN plans p ON s.plan_id = p.plan_id\n    WHERE c.customer_id = ?\n    ORDER BY s.primary_subscription_flag DESC, s.activation_date DESC;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id,))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_churn_risk(customer_id: int) -> str:\n    \"\"\"Get churn score, churn risk level, main risk reason, and recommended retention action for a customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        ch.score_month,\n        ch.churn_score,\n        ch.risk_level,\n        ch.main_risk_reason,\n        ch.recommended_action\n    FROM customer_churn_scores ch\n    JOIN customers c ON ch.customer_id = c.customer_id\n    WHERE c.customer_id = ?;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id,))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_complaints(customer_id: int, limit: int = 5) -> str:\n    \"\"\"Get recent customer complaints including category, description, severity, status, and compensation amount.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        complaint_date,\n        complaint_category,\n        complaint_description,\n        severity,\n        status,\n        resolved_date,\n        compensation_amount_jod\n    FROM complaints\n    WHERE customer_id = ?\n    ORDER BY complaint_date DESC\n    LIMIT ?;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id, limit))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_billing_summary(customer_id: int, limit: int = 5) -> str:\n    \"\"\"Get recent invoice and billing summary including invoice dates, total amount, payment status, and days overdue for a customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        i.invoice_id,\n        i.billing_period_start,\n        i.billing_period_end,\n        i.issue_date,\n        i.due_date,\n        i.total_amount_jod,\n        i.payment_status,\n        i.days_overdue\n    FROM customers c\n    JOIN accounts a ON c.customer_id = a.customer_id\n    JOIN invoices i ON a.account_id = i.account_id\n    WHERE c.customer_id = ?\n    ORDER BY i.issue_date DESC\n    LIMIT ?;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id, limit))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_usage_summary(customer_id: int) -> str:\n    \"\"\"Get customer data usage summary including sessions, total data used in GB, data cost, and last data session for a customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        s.subscription_id,\n        s.service_type,\n        COUNT(d.session_id) AS total_data_sessions,\n        ROUND(SUM(d.data_used_mb) / 1024.0, 2) AS total_data_used_gb,\n        ROUND(SUM(d.cost_jod), 2) AS total_data_cost_jod,\n        MAX(d.session_start_time) AS last_data_session\n    FROM customers c\n    JOIN subscriptions s ON c.customer_id = s.customer_id\n    LEFT JOIN data_usage_sessions d ON s.subscription_id = d.subscription_id\n    WHERE c.customer_id = ?\n    GROUP BY c.customer_id, c.full_name, s.subscription_id, s.service_type\n    ORDER BY total_data_used_gb DESC;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id,))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.tool()\ndef get_customer_value_segment(customer_id: int) -> str:\n    \"\"\"Get customer value segment, ARPU, six-month revenue, and lifetime months for a Zain Jordan customer ID.\"\"\"\n    conn = get_connection()\n    query = \"\"\"\n    SELECT \n        c.customer_id,\n        c.full_name,\n        v.segment_month,\n        v.arpu_jod,\n        v.total_revenue_6m_jod,\n        v.value_segment,\n        v.lifetime_months\n    FROM customer_value_segments v\n    JOIN customers c ON v.customer_id = c.customer_id\n    WHERE c.customer_id = ?;\n    \"\"\"\n    df = pd.read_sql_query(query, conn, params=(customer_id,))\n    conn.close()\n    return df_to_text(df)\n\n@mcp.resource(\"schema://tables\")\ndef list_database_tables() -> str:\n    \"\"\"List database tables as a read-only MCP resource.\"\"\"\n    conn = get_connection()\n    df = pd.read_sql_query(\"\"\"\n    SELECT name \n    FROM sqlite_master \n    WHERE type = 'table'\n    ORDER BY name;\n    \"\"\", conn)\n    conn.close()\n    return df_to_text(df, max_rows=100)\n\n@mcp.prompt()\ndef customer_risk_analysis_prompt(customer_id: int) -> str:\n    \"\"\"Reusable prompt for customer risk analysis.\"\"\"\n    return f\"\"\"\nAnalyze Zain Jordan customer {customer_id} using available MCP tools.\nCheck profile, plan, churn risk, complaints, billing, usage, and value segment.\nThen recommend the next best action.\n\"\"\"\n\nif __name__ == \"__main__\":\n    mcp.run(transport=\"stdio\")\n"

server_code = server_code.replace("__DB_PATH_LITERAL__", repr(str(db_path_obj)))
SERVER_FILE.write_text(server_code, encoding="utf-8")

print("MCP server file created:")
print(SERVER_FILE)
print("\nFirst 30 lines:")
print("\n".join(SERVER_FILE.read_text(encoding="utf-8").splitlines()[:30]))


In [ ]:
import py_compile

py_compile.compile(str(SERVER_FILE), doraise=True)

print("MCP server syntax check passed.")


# 7. Connect to MCP Server with LangChain

We use `MultiServerMCPClient`.

The client launches the local MCP server as a subprocess using stdio transport.


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "zain_customer": {
            "transport": "stdio",
            "command": "python",
            "args": [str(SERVER_FILE)],
        }
    }
)

print("MCP client configured.")


# 8. Load MCP Tools

This cell uses top-level `await`, which works in Google Colab and Jupyter.


In [ ]:
mcp_tools = await client.get_tools()

print("Number of MCP tools loaded:", len(mcp_tools))

for t in mcp_tools:
    print("Tool:", t.name)
    print("Description:", t.description)
    print("-" * 100)


# 9. Test MCP Tools Directly

Before giving MCP tools to an agent, test them directly.


In [ ]:
profile_tool = next(t for t in mcp_tools if t.name == "get_customer_profile")
churn_tool = next(t for t in mcp_tools if t.name == "get_customer_churn_risk")
billing_tool = next(t for t in mcp_tools if t.name == "get_customer_billing_summary")

print("PROFILE")
print(await profile_tool.ainvoke({"customer_id": 42}))

print("\\nCHURN")
print(await churn_tool.ainvoke({"customer_id": 42}))

print("\\nBILLING")
print(await billing_tool.ainvoke({"customer_id": 42, "limit": 3}))


# 10. Create an MCP-Powered LangChain Agent

The agent does not directly know the database. It only sees tools loaded from the MCP server.


In [ ]:
from langchain.agents import create_agent

MODEL_NAME = "openai:gpt-4.1-mini"

mcp_agent_prompt = """
You are a professional Zain Jordan customer-care AI assistant.

You have access to customer database tools through MCP.

Rules:
1. Use MCP tools whenever customer data is needed.
2. Do not guess customer facts.
3. If a tool returns no records, say the data was not available.
4. Do not mention internal technical details to the customer.
5. Keep answers structured and business-friendly.

For full customer analysis, include:
- Customer Summary
- Plan Summary
- Churn Risk
- Value Segment
- Billing Summary
- Complaints
- Usage Summary
- Recommended Next Action
- Suggested Customer-Care Message
"""

mcp_agent = create_agent(
    model=MODEL_NAME,
    tools=mcp_tools,
    system_prompt=mcp_agent_prompt,
)

print("MCP-powered LangChain agent created.")


In [ ]:
def extract_final_text(result):
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\\n".join(parts)

    return str(last_message)


async def run_mcp_agent(question: str):
    result = await mcp_agent.ainvoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


# 11. Demo 1: Simple MCP Agent Question


In [ ]:
question = "Who is customer 42? Use MCP tools."

answer = await run_mcp_agent(question)
print(answer)


# 12. Demo 2: Churn Risk via MCP Tools


In [ ]:
question = """
Use MCP tools to check the churn risk for customer 42.
Explain the risk level, main risk reason, and recommended action.
"""

answer = await run_mcp_agent(question)
print(answer)


# 13. Main Demo: Full Customer 360 Analysis via MCP


In [ ]:
question = """
Use the Zain Jordan MCP tools to analyze customer 42.

Check:
1. customer profile
2. current plan
3. churn risk
4. complaints
5. billing status
6. usage summary
7. value segment

Then recommend the next best action and draft a short professional customer-care message.
"""

answer = await run_mcp_agent(question)
print(answer)


# 14. Demo 4: Compare Two Customers with MCP Tools


In [ ]:
question = """
Use MCP tools to compare customer 25 and customer 42.

Check profile, churn risk, value segment, complaints, and billing.
Which customer should the care team prioritize first and why?
"""

answer = await run_mcp_agent(question)
print(answer)


# 15. Optional: Show MCP Agent Tool Calls

This helps trainers debug what tools were called.


In [ ]:
debug_result = await mcp_agent.ainvoke({
    "messages": [
        {"role": "user", "content": "Use MCP tools to check billing and churn risk for customer 42."}
    ]
})

for i, message in enumerate(debug_result["messages"], start=1):
    print(f"--- Message {i} ---")
    print(message)
    print()


# 16. Exercise 1: MCP Tool Mapping

| User Request | Best MCP Tool |
|---|---|
| Who is customer 42? | `get_customer_profile` |
| What plan does customer 42 have? | `get_customer_plan` |
| Is customer 42 likely to leave? | `get_customer_churn_risk` |
| Did customer 42 complain recently? | `get_customer_complaints` |
| Does customer 42 have unpaid bills? | `get_customer_billing_summary` |
| How much data did customer 42 use? | `get_customer_usage_summary` |
| Is customer 42 high value? | `get_customer_value_segment` |


# 17. Exercise 2: Analyze Different Customers


In [ ]:
exercise_question = """
Use MCP tools to analyze customer 10.
Check profile, plan, churn risk, billing, complaints, usage, and value segment.
Recommend the next best action.
"""

answer = await run_mcp_agent(exercise_question)
print(answer)


In [ ]:
exercise_question = """
Use MCP tools to analyze customer 100.
Check profile, plan, churn risk, billing, complaints, usage, and value segment.
Recommend the next best action.
"""

answer = await run_mcp_agent(exercise_question)
print(answer)


# 18. MCP vs SQL vs RAG

| Request | Best Pattern |
|---|---|
| Get customer 42 profile | MCP Tool |
| Which cities have most churn? | SQL Agent |
| Which offer suits roaming customers? | RAG |
| Analyze customer 42 end-to-end | Multi-Agent + MCP Tools |
| Draft a response message | Care Message Agent / LLM |


# 19. Capstone MCP Design

For your capstone project, define:

1. Which tools should become MCP tools?
2. What are the inputs?
3. What are the outputs?
4. Who is the business user?
5. What safety controls are needed?

Example:

| Tool | Input | Output | User |
|---|---|---|---|
| `get_customer_churn_risk` | customer_id | churn score, reason, action | Retention team |
| `get_customer_billing_summary` | customer_id | invoice/payment summary | Billing support |
| `get_customer_value_segment` | customer_id | ARPU, revenue, value segment | Customer care manager |


# 20. Common MCP Mistakes

1. Exposing too many tools too early.
2. Poor tool names like `get_data`.
3. Unsafe database access.
4. Confusing MCP with RAG.
5. Building MCP before tools are understood.


# 21. Safety and Governance Notes

For training, we are using:

- synthetic database
- local SQLite file
- selected read-only tools
- limited outputs

For production, consider:

1. Read-only credentials.
2. Tool-level permissions.
3. Column restrictions for sensitive data.
4. Input validation.
5. Audit logging.
6. Output limits.
7. Human review for sensitive actions.
8. Authentication for remote MCP servers.


# 22. What We Built Today

In Class 6, we built:

1. A local MCP server file.
2. Zain Jordan database tools exposed through MCP.
3. A LangChain MCP client using `MultiServerMCPClient`.
4. MCP tools loaded as LangChain-compatible tools.
5. A LangChain agent powered by MCP tools.
6. Full customer-care analysis using MCP.
7. Capstone MCP mapping exercises.

## Trainer Closing Script

Today, we moved from prototype tools to enterprise-style tool integration.

The key idea is:

> MCP standardizes how AI applications connect to tools and data.

This prepares us for real enterprise AI systems where tools, databases, and workflows live outside the agent and are connected through standard interfaces.
